# Real CASAS multi-room occupancy traces

This notebook demonstrates the dataset-generation script (`data.py`) behind the **Real CASAS multi-room occupancy traces** artifact.

The original script standardizes real, room-level residential occupancy data from the [WSU CASAS smart-home archive](https://zenodo.org/records/17180309) (Zenodo record 17180309) into the `exp_sel_data_out.json` schema used across this project. It covers four real houses (Aruba, Cairo, Milan, Tulum): raw motion-sensor ON/OFF event logs are binned into 96 fifteen-minute-per-day binary occupancy vectors per (house, day, room). Each example's `input` is one day's 96-bin occupancy vector for one room, and the `output` is that same vector shifted one bin ahead — a next-bin-occupied forecasting target designed for a **PreHeat**-style per-room K-NN Hamming-distance predictor that pre-heats a room just before it is likely to become occupied (the practical energy-saving method this dataset validates).

This demo notebook:
1. Installs dependencies and loads a small curated subset of the raw CASAS data (`mini_demo_data.json`, in the exact raw format `data.py`'s `build_examples()` consumes).
2. Runs `data.py`'s **original, unmodified** transformation logic (`build_examples`) to produce standardized examples.
3. Demonstrates the PreHeat-style K-NN Hamming-distance next-bin-occupancy predictor the dataset was built to support, and reports its held-out accuracy — a small validation of the practical heating-energy-saving method the dataset targets.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru is used by the original data.py — not pre-installed on Colab
_pip('loguru==0.7.3')

# numpy / matplotlib are pre-installed on Colab; install locally to match Colab's exact versions
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'matplotlib==3.10.0')

In [ ]:
# --- imports (original data.py imports, plus numpy/matplotlib for the demo) ---
import json
import sys
from pathlib import Path

from loguru import logger

import numpy as np
import matplotlib.pyplot as plt

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

## Load the data

`mini_demo_data.json` is a curated 100-row subset of the real, raw CASAS Aruba house data — 60 days of room `M003` and 40 days of room `M002` — kept in **exactly** the raw `{"metadata_fold": ..., "rows": [...]}` shape that `data.py`'s original `build_examples()` function consumes (one `full_casas_<house>.json` file, unmodified format). We try the GitHub-hosted copy first (works once this repo is published), falling back to the local file (works right now, and in this session).

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-ba45a6-transition-sequence-pre-heating-beats/main/round-1/dataset-1/demo/mini_demo_data.json"
import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
payload = load_data()
print("rooms in this raw payload:", payload["metadata_fold"]["rooms"][:8], "...")
print("n_rows:", len(payload["rows"]))
print("example row keys:", list(payload["rows"][0].keys()))

## Config

All tunable parameters, gathered in one place. Values start small (K=1, few held-out days) so the demo runs in seconds; increase them to scale toward the full artifact.

In [ ]:
# --- data.py's own parameter, unchanged ---
MAX_TRANSITIONS_PER_EXAMPLE = 40  # caps pathological high-traffic sensors (e.g. hallway doorways)

# --- PreHeat K-NN validation demo parameters (start minimal, then scale up) ---
K_NEIGHBORS = 1          # number of nearest similar days to average (PreHeat uses K=5 at full scale)
N_HELDOUT_DAYS = 3       # number of most-recent days per room held out for evaluation
# ORIGINAL FULL-SCALE VALUES (commented out — use the full dataset for these):
# K_NEIGHBORS = 5
# N_HELDOUT_DAYS = 20

## Standardize the raw data (original `data.py` logic, unmodified)

`_minute_of_day` and `build_examples` are copied verbatim from `data.py`. `build_examples` turns each raw `(day_id, room_id)` row into a standardized example: the 96-bin `input`/`output` occupancy vectors plus the room's transitions for that day (capped at `MAX_TRANSITIONS_PER_EXAMPLE`).

In [ ]:
def _minute_of_day(ts: str) -> str:
    # SPHERE timestamps look like "seq12_t345.67"; CASAS are ISO datetimes.
    if "T" in ts:
        return ts.split("T", 1)[1][:8]
    return ts


def build_examples(dataset_id: str, payload: dict) -> list[dict]:
    meta = payload["metadata_fold"]
    transitions_by_day = meta.get("transitions_by_day", {})
    transition_fields = meta.get("transition_fields", ["from_room", "to_room", "timestamp", "dwell_minutes"])

    examples = []
    for row in payload["rows"]:
        day_id = row["day_id"]
        room_id = row["room_id"]
        day_transitions = transitions_by_day.get(day_id, [])
        # compact [from_room, to_room, minute_of_day, dwell_minutes] tuples (timestamps -> minute-of-day int)
        room_transitions = [
            [t[0], t[1], _minute_of_day(t[2]), t[3]]
            for t in day_transitions
            if t[0] == room_id or t[1] == room_id
        ][:MAX_TRANSITIONS_PER_EXAMPLE]

        example = {
            "input": json.dumps(row["input"]),
            "output": json.dumps(row["output"]["next_bin_occupied"]),
            "metadata_dataset_source": meta["source"],
            "metadata_house_id": meta["house_id"],
            "metadata_day_id": day_id,
            "metadata_room_id": room_id,
            "metadata_weekday_weekend": row["weekday_weekend"],
            "metadata_real_or_synthetic": meta["real_or_synthetic"],
            "metadata_adjacency_provenance": meta.get("adjacency_provenance", "unknown"),
            "metadata_room_transitions": room_transitions,
            "metadata_task_type": "binary_sequence_forecasting",
            "metadata_n_classes": 2,
            "metadata_bin_minutes": 15,
        }
        examples.append(example)
    return examples


dataset_id = payload["metadata_fold"]["dataset_id"]
examples = build_examples(dataset_id, payload)
logger.info(f"{dataset_id}: {len(examples)} standardized examples")
print(json.dumps({k: (v if k not in ("input", "output") else json.loads(v)[:10]) for k, v in examples[0].items()}, indent=2)[:1200])

## Validate the practical method: PreHeat's per-room K-NN Hamming-distance predictor

This dataset was built (per its description) to feed a **PreHeat**-style predictor: for a target day/room, find the `K_NEIGHBORS` most similar historical days for that same room (matched on `weekday_weekend`, nearest by Hamming distance on the 96-bin `input` vector), and predict next-bin occupancy as the majority vote of those neighbors' `output` vectors. A smart thermostat would use this per-bin prediction to pre-heat a room only in the 15-minute bins it expects to be occupied, cutting wasted heating of empty rooms.

For each room, we hold out the last `N_HELDOUT_DAYS` days as a test set and use all earlier days (matched on weekday/weekend) as the K-NN pool, then measure next-bin prediction accuracy against a trivial baseline (always-predict-"occupied-in-the-most-common-state").

In [ ]:
# group standardized examples by room, sorted by day_id (chronological)
by_room = {}
for ex in examples:
    by_room.setdefault(ex["metadata_room_id"], []).append(ex)
for room_id in by_room:
    by_room[room_id].sort(key=lambda e: e["metadata_day_id"])


def hamming(a, b):
    return sum(1 for x, y in zip(a, b) if x != y)


def knn_predict(target_input, pool, k):
    """PreHeat-style predictor: K nearest days (by Hamming distance on the input
    occupancy vector) vote bin-by-bin on next-bin occupancy."""
    dists = sorted(pool, key=lambda e: hamming(target_input, json.loads(e["input"])))[:k]
    neighbor_outputs = [json.loads(e["output"]) for e in dists]
    n_bins = len(target_input)
    pred = []
    for bin_idx in range(n_bins):
        votes = sum(o[bin_idx] for o in neighbor_outputs)
        pred.append(1 if votes * 2 >= len(neighbor_outputs) else 0)
    return pred


results_per_room = {}
for room_id, room_examples in by_room.items():
    if len(room_examples) <= N_HELDOUT_DAYS:
        continue  # not enough history to hold out test days for this room
    train, test = room_examples[:-N_HELDOUT_DAYS], room_examples[-N_HELDOUT_DAYS:]
    correct_knn, correct_baseline, total_bins = 0, 0, 0
    for test_ex in test:
        # match on weekday/weekend, as PreHeat's K=5 nearest-similar-day matching does
        pool = [e for e in train if e["metadata_weekday_weekend"] == test_ex["metadata_weekday_weekend"]] or train
        target_input = json.loads(test_ex["input"])
        target_output = json.loads(test_ex["output"])
        pred = knn_predict(target_input, pool, K_NEIGHBORS)
        baseline_pred = target_input  # naive baseline: next bin = same as current bin
        correct_knn += sum(1 for p, t in zip(pred, target_output) if p == t)
        correct_baseline += sum(1 for p, t in zip(baseline_pred, target_output) if p == t)
        total_bins += len(target_output)
    results_per_room[room_id] = {
        "n_test_days": len(test),
        "knn_accuracy": correct_knn / total_bins,
        "baseline_accuracy": correct_baseline / total_bins,
    }

results_per_room

## Results

A readable summary table of the K-NN predictor vs. the naive baseline, plus a plot of one example day's real occupancy trace against the predictor's next-bin forecast.

In [ ]:
print(f"{'room':<8}{'test_days':<12}{'knn_accuracy':<16}{'baseline_accuracy':<18}")
for room_id, r in results_per_room.items():
    print(f"{room_id:<8}{r['n_test_days']:<12}{r['knn_accuracy']:<16.3f}{r['baseline_accuracy']:<18.3f}")

mean_knn = np.mean([r["knn_accuracy"] for r in results_per_room.values()])
mean_baseline = np.mean([r["baseline_accuracy"] for r in results_per_room.values()])
print(f"\
mean next-bin-occupancy accuracy: KNN(K={K_NEIGHBORS})={mean_knn:.3f}  vs  same-bin baseline={mean_baseline:.3f}")

# --- plot: real occupancy trace vs. KNN forecast for one held-out day ---
room_id = next(iter(results_per_room))
test_ex = by_room[room_id][-1]
train = by_room[room_id][:-N_HELDOUT_DAYS]
pool = [e for e in train if e["metadata_weekday_weekend"] == test_ex["metadata_weekday_weekend"]] or train
target_input = json.loads(test_ex["input"])
target_output = json.loads(test_ex["output"])
pred = knn_predict(target_input, pool, K_NEIGHBORS)

bins = np.arange(len(target_output))
fig, ax = plt.subplots(figsize=(10, 3))
ax.step(bins, target_output, where="post", label="actual next-bin occupancy", linewidth=2)
ax.step(bins, [p - 1.2 for p in pred], where="post", label="KNN predicted next-bin occupancy (offset)", linewidth=2)
ax.set_xlabel("15-minute bin of day")
ax.set_yticks([])
ax.set_title(f"Room {room_id}, day {test_ex['metadata_day_id']} ({test_ex['metadata_weekday_weekend']}): actual vs. PreHeat-style KNN forecast")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()